In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
_os.environ['AWS_REGION'] = 'us-west-2'
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker Clarify E2E Test

Simple end-to-end test for the Clarify utils implementation

In [2]:
import sys
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib
import boto3

# Add the clarify utils to path
# sys.path.insert(0, '/Users/mollyhe/Documents/SageMaker/sagemaker-python-sdk-staging-molly/sagemaker_utils/src')

from sagemaker.core.clarify import (
    SageMakerClarifyProcessor,
    DataConfig,
    BiasConfig,
    ModelConfig,
    SHAPConfig
)
from sagemaker.core.helper.session_helper import Session,get_execution_role
role = get_execution_role()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


## 1. Create Sample Data

In [3]:
# Create synthetic dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42
)

# Add a sensitive feature (simulating gender: 0=female, 1=male)
sensitive_feature = np.random.binomial(1, 0.4, size=X.shape[0])
X = np.column_stack([X, sensitive_feature])

# Create DataFrame
feature_names = [f'feature_{i}' for i in range(10)] + ['gender']
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print(f"Dataset shape: {df.shape}")
print(f"Target distribution: {df['target'].value_counts()}")
print(f"Gender distribution: {df['gender'].value_counts()}")

Dataset shape: (1000, 12)
Target distribution: target
1    503
0    497
Name: count, dtype: int64
Gender distribution: gender
0.0    593
1.0    407
Name: count, dtype: int64


## 2. Train Simple Model

In [4]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('target', axis=1), df['target'], test_size=0.2, random_state=42
)

# Train model
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X_train, y_train)

print(f"Model accuracy: {model.score(X_test, y_test):.3f}")

Model accuracy: 0.935


## 3. Upload Data to S3

In [5]:
# Setup S3 paths
session = Session()
bucket = session.default_bucket()
prefix = 'clarify-test'

# Save test data (without target for inference)
test_data = X_test.copy()
test_data['target'] = y_test
test_data.to_csv('/tmp/test_data.csv', index=False)

# Save model
joblib.dump(model, '/tmp/model.joblib')

# Upload to S3
s3_client = boto3.client('s3', region_name='us-west-2')
s3_client.upload_file('/tmp/test_data.csv', bucket, f'{prefix}/data/test_data.csv')
s3_client.upload_file('/tmp/model.joblib', bucket, f'{prefix}/model/model.joblib')

data_uri = f's3://{bucket}/{prefix}/data/test_data.csv'
output_uri = f's3://{bucket}/{prefix}/output'

print(f"Data uploaded to: {data_uri}")
print(f"Output will be saved to: {output_uri}")

Data uploaded to: s3://sagemaker-us-west-2-674622101542/clarify-test/data/test_data.csv
Output will be saved to: s3://sagemaker-us-west-2-674622101542/clarify-test/output


## 4. Configure Clarify

In [6]:
# Data configuration
data_config = DataConfig(
    s3_data_input_path=data_uri,
    s3_output_path=output_uri,
    label='target',
    headers=list(test_data.columns),
    dataset_type='text/csv'
)

# Bias configuration
bias_config = BiasConfig(
    label_values_or_threshold=[1],  # Positive class
    facet_name='gender',
    facet_values_or_threshold=[1]   # Male as sensitive group
)

# SHAP configuration
shap_config = SHAPConfig(
    baseline=None,  # Auto-generate baseline
    num_samples=10,  # Small number for quick test
    agg_method='mean_abs'
)

print("Configurations created successfully")

Configurations created successfully


## 5. Create Clarify Processor

In [7]:
# Create Clarify processor
clarify_processor = SageMakerClarifyProcessor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    sagemaker_session=session
)

print(f"Clarify processor created with role: {role}")

[07/21/26 07:48:43] INFO     Ignoring unnecessary instance type: None.                            ]8;id=5751866;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=5751867;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py#535\535]8;;\

Clarify processor created with role: arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole


## 6. Run Pre-training Bias Analysis

In [8]:
# Run pre-training bias analysis (no model needed)
try:
    clarify_processor.run_pre_training_bias(
        data_config=data_config,
        data_bias_config=bias_config,
        methods=['CI', 'DPL'],  # Class Imbalance and Difference in Positive Proportions
        wait=False,  # Don't wait for completion in test
        logs=False
    )
    print("✅ Pre-training bias analysis job submitted successfully")
except Exception as e:
    print(f"❌ Pre-training bias analysis failed: {str(e)}")

[07/21/26 07:48:44] INFO     Analysis Config: {'dataset_type': 'text/csv', 'headers':              ]8;id=5751874;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/clarify/__init__.py\__init__.py]8;;\:]8;id=5751875;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/clarify/__init__.py#1992\1992]8;;\
                             ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4',                     
                             'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9',                      
                             'gender', 'target'], 'label': 'target', 'label_values_or_threshold':                  
                             [1], 'facet': [{'name_or_index': 'gender', 'value_or_threshold':                      
                             [1]}], 'methods': {'report': {'name': 'report', 'title': 'Analysis                    
                             Report'}, 'pre_training_bias': {'methods': ['CI', 'DPL']}}}                           

[07/21/26 07:48:45] INFO     Creating processing-job with name                                    ]8;id=5751882;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/processing.py\processing.py]8;;\:]8;id=5751883;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/processing.py#614\614]8;;\
                             Clarify-Pretraining-Bias-2026-07-21-07-48-44-430                                      

✅ Pre-training bias analysis job submitted successfully


In [9]:
# You can go to SageMaker AI console -> Processing jobs and check the job status
# Or you can run the below command
# Note that it takes ~5min for the job to be complete

response = session.sagemaker_client.describe_processing_job(ProcessingJobName=clarify_processor._current_job_name)
print(f"Status: {response['ProcessingJobStatus']}")

Status: InProgress


## 7. Test Configuration Generation

In [10]:
# Test the internal config generation
from sagemaker.core.clarify import _AnalysisConfigGenerator

try:
    # Generate bias config
    bias_analysis_config = _AnalysisConfigGenerator.bias_pre_training(
        data_config=data_config,
        bias_config=bias_config,
        methods=['CI', 'DPL']
    )
    
    print("✅ Bias analysis config generated successfully")
    print(f"Config keys: {list(bias_analysis_config.keys())}")
    
    # Validate config structure
    required_keys = ['dataset_type', 'label_values_or_threshold', 'facet', 'methods']
    missing_keys = [key for key in required_keys if key not in bias_analysis_config]
    
    if missing_keys:
        print(f"❌ Missing required keys: {missing_keys}")
    else:
        print("✅ All required keys present in config")
        
except Exception as e:
    print(f"❌ Config generation failed: {str(e)}")

✅ Bias analysis config generated successfully
Config keys: ['dataset_type', 'headers', 'label', 'label_values_or_threshold', 'facet', 'methods']
✅ All required keys present in config


## 8. Test Schema Validation

In [11]:
# Test schema validation
from sagemaker.core.clarify import ANALYSIS_CONFIG_SCHEMA_V1_0

try:
    # Validate the generated config
    ANALYSIS_CONFIG_SCHEMA_V1_0.validate(bias_analysis_config)
    print("✅ Schema validation passed")
except Exception as e:
    print(f"❌ Schema validation failed: {str(e)}")

✅ Schema validation passed
